In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

import os
import pickle
import random
import sys

from scipy.stats import ttest_ind
from torkin import TorKin

from config import Config
cfg = Config()

plt.rcParams['font.size'] = 20
plt.rc('font', size=20)
plt.rc('legend', fontsize=16)
label_fontsize = 16

In [2]:
root_dir = "/home/arash/catkin_ws/src/feedback_controller/fbc/neural_network/results/"
ds_name = "trajs:72_blocks:3_triangle_v_scarce" #"trajs:500_blocks:3_random" #"triangle"
# ds_name = "trajs:500_blocks:3_random"
# ds_name = "triangle"
ds_type = "extrap_0.85" #"interp_0.85" #"interp_0.85"
# ds_type = "interp_0.95"
# noise = "0.008_noise"
model_name = f"on_2+2l_lat:sub-nvel_1e-05" #_freeze

model_params_list = [6.037, 12.053, 24.085] #, 48.149] #, 36.117]
episode_no = 4000
# episode_no = 12000

kin = TorKin()

No data file named /home/erhan/catkin_ws/src/erhtor_work/tormain/Resources/larm60.txt
bodypart structure for larm is constructed.
No data file named /home/erhan/catkin_ws/src/erhtor_work/tormain/Resources/rarm60.txt
bodypart structure for rarm is constructed.
bodypart structure for head is constructed.


# Latent Representations

In [3]:
def get_file(model_dir, is_not_abl, file_name):
    file_dir = os.path.join(model_dir, f"{model_name}_{is_not_abl}_3l_base")
    file_path = os.path.join(file_dir, f"{file_name}")
    with open(file_path, 'rb') as f:
        file = pickle.load(f)
    return file

In [4]:
# num_params = model_params_list[0]

# model_dir = os.path.join(root_dir, f"{ds_name}_{num_params}K_{ds_type}/ep:{episode_no}/")
# # latent_reps = get_file(model_dir, True, "all_latent_reps")
# latent_reps_abl = get_file(model_dir, False, "all_latent_reps")
# # states_dnfc = get_file(model_dir, True, "all_states_dnfc")
# states_abl = get_file(model_dir, False, "all_states_dnfc")

In [5]:
# def change_indices_per_col(X):
#     """
#     X: array of shape (n, d)
#     returns: dict {col_index: np.array of indices where that column changes}
#              indices are in [1..n-1] because each change is compared to the previous row
#     """
#     X = np.asarray(X)
#     # shape: (n-1, d) — True where X[t] != X[t-1]
#     changed = X[1:] != X[:-1]
#     # collect indices per column (+1 to map to the row where the change occurs)
#     return {j: (np.where(changed[:, j])[0] + 1) for j in range(X.shape[1])}

In [6]:
# array = np.array(latent_reps_abl[3])
# change_indices_per_col(array)

In [7]:
# print(array[190])
# print(array[191])
# print(array[192])
# print(array[193])

In [8]:
# print(len(latent_reps))
# print(len(latent_reps[0]))
# print(len(latent_reps[0][0]))

In [9]:
def get_csv(model_dir, is_not_abl):
    file_dir = os.path.join(model_dir, f"{model_name}_{is_not_abl}_3l_base")
    file_path = os.path.join(file_dir, "perf.csv")
    df = pd.read_csv(file_path)  
    return df

def add_train_round(df):
    df['train_round'] = df.groupby('eps_num').cumcount() + 1
    return df

In [10]:
def form_plot(axes, lat_rep_transp, lat_rep_abl_transp, 
              states_transp, states_abl_transp,
              colors, i_st, i_en):
    
    time_steps = range(len(lat_rep_transp[0]))
    idx = 0
    for i in range(i_st, i_en):
        ax = axes[idx]
        if i < 7: label_st = f'Jnt.{i + 1}'
        else: label_st = f'Vel.{(i%7) + 1}'

        ax.plot(time_steps, lat_rep_transp[i], 
                 label=f'Dim{i + 1}', color=colors[i%7])
        ax.plot(time_steps, lat_rep_abl_transp[i], 
                 label=f'Dim{i + 1}_0', color=colors[(i+1)%7])
        ax.plot(time_steps, states_transp[i], 
                 label=label_st, color=colors[(i+2)%7]) #<, linestyle='dashed')
        ax.plot(time_steps, states_abl_transp[i], 
                 label=f'{label_st}_0', color=colors[(i+3)%7], linestyle='dotted')
        
        idx += 1
        # ax.set_ylabel(f'Dim {i + 1}')
        ax.legend()
        ax.grid(True)

    axes[-1].set_xlabel('Time Steps')
    return axes


def save_plot(model_dir, df_perf, plt_name):
    plot_dir = os.path.join(model_dir, f'latent_reps_2')
    if os.path.exists(plot_dir):
        pass
        # print("Latent dir. already exists. Are you trying to generate it again?")
        # sys.exit(1)
    else:
        os.makedirs(plot_dir)
    plot_path = os.path.join(plot_dir, 
                             f"eps:{df_perf['eps_num']}_train:{df_perf['train_round']-1}_{plt_name}.png")
    plt.savefig(plot_path)
    plt.close()


def plot_latent_reps(latent_reps, latent_reps_abl, states, states_abl,
                     df_perf, df_perf_abl, model_dir):
    # Transpose the data to separate each joint
    lat_rep_transp = list(zip(*latent_reps))
    lat_rep_abl_transp = list(zip(*latent_reps_abl))
    states_transp = list(zip(*states))
    states_abl_transp = list(zip(*states_abl))
    
    colors = [
        'blue', 'green', 'red', 'cyan', 'magenta', #'yellow', 
        'black', 'orange', 'purple', 'brown', 'pink', 'lime', 
        'teal', 'gold'
    ]
    
    # Create a figure with 7 subplots arranged vertically
    fig, axes = plt.subplots(7, 1, figsize=(14, 14), sharex=True)
    fig.tight_layout(pad=0.0)  #Adjust spacing between subplots
    axes = form_plot(axes, lat_rep_transp, lat_rep_abl_transp, 
                        states_transp, states_abl_transp,
                        colors, i_st=0 , i_en=7)
    fig.suptitle(f"LFEL: {df_perf['dnfc_succ']}, LFEL_0: {df_perf_abl['dnfc_succ']}")
    save_plot(model_dir, df_perf, "jnt")

    fig, axes = plt.subplots(7, 1, figsize=(14, 14), sharex=True)
    fig.tight_layout(pad=0.0)  #Adjust spacing between subplots
    axes = form_plot(axes, lat_rep_transp, lat_rep_abl_transp, 
                        states_transp, states_abl_transp,
                        colors, i_st=7 , i_en=14)
    fig.suptitle(f"LFEL: {df_perf['dnfc_succ']}, LFEL_0: {df_perf_abl['dnfc_succ']}")
    save_plot(model_dir, df_perf, "vel")

In [11]:
for num_params in model_params_list:

    model_dir = os.path.join(root_dir, f"{ds_name}_{num_params}K_{ds_type}/ep:{episode_no}/")
    latent_reps = get_file(model_dir, True, "all_latent_reps")
    latent_reps_abl = get_file(model_dir, False, "all_latent_reps")
    # latent_reps = latent_reps_abl
    states_dnfc = get_file(model_dir, True, "all_states_dnfc")
    states_abl = get_file(model_dir, False, "all_states_dnfc")
    # states_dnfc = states_abl

    df_perf = get_csv(model_dir, True)
    df_perf = add_train_round(df_perf)
    df_perf_abl = get_csv(model_dir, False)
    df_perf_abl = add_train_round(df_perf_abl)
    # df_perf = df_perf_abl

    for i_eps in range(len(latent_reps)):
        plot_latent_reps(latent_reps[i_eps], 
                         latent_reps_abl[i_eps], 
                         states_dnfc[i_eps],
                         states_abl[i_eps], 
                         df_perf.loc[i_eps], df_perf_abl.loc[i_eps], 
                         model_dir)

# Model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

from testers import Tester
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
tester = Tester()
i_train = 1
epoch_no = 4000
tester.load_model(i_train, epoch_no, 
                  tester.config.use_custom_loss, 'low')

In [ ]:
tester.model

In [ ]:
joint_size = tester.joint_size
state_size = tester.state_size
step_size = tester.step_size
target_size = tester.target_size
onehot_size = tester.onehot_size
traj_step_size = 900

## Example

In [ ]:
eps_num = 0
timestep_no = 270

elem = tester.dataset[eps_num]
print(elem.shape)
state = torch.tensor(elem[timestep_no][step_size : 
                                step_size + state_size].tolist()
                                ).to(device)

# milestones = tester.get_changes_indexes(eps_num)
one_hot = elem[timestep_no][step_size+state_size+target_size : 
                    step_size+state_size+target_size+onehot_size].tolist()
goal = elem[timestep_no][step_size+state_size : 
                step_size+state_size+target_size].tolist()

goal_tensor = torch.tensor(goal+one_hot).to(device)
goal_nn = torch.unsqueeze(goal_tensor, 0)
state_nn = torch.unsqueeze(state, 0)

print(goal)
print(one_hot)
print(goal_nn)

In [ ]:
tester.model.eval()
velocities_tensor, x_des, _ = tester.model(goal_nn, state_nn)
x_des

In [ ]:
tester.model.eval()
velocities_tensor, x_des, _ = tester.model(goal_nn, state_nn)
x_des

In [ ]:
for name, tensor in tester.model.enc1.state_dict().items():
    print(name, tensor.shape)
    if not "bias" in name:
        print(tensor[:2, :])
    print("===")

## Loop

In [ ]:
def form_plot(all_latent_transp, i_st, i_en, colors):
    # Create a figure with 7 subplots arranged vertically
    fig, axes = plt.subplots(7, 1, figsize=(14, 14), sharex=True)
    fig.tight_layout(pad=0.0)  #Adjust spacing between subplots
    # axes = form_plot(axes, lat_rep_transp, colors, i_st=0 , i_en=7)
    idy = 0
    for lat_rep_transp in all_latent_transp:
        time_steps = range(len(lat_rep_transp[0]))
        idx = 0
        for i in range(i_st, i_en):
            ax = axes[idx]
            if i < 7: label_st = f'Jnt.{i + 1}'
            else: label_st = f'Vel.{(i%7) + 1}'

            ax.scatter(time_steps, lat_rep_transp[i], 
                    label=f'Dim{i + 1}_{idy}', color=colors[idy])
            idx += 1
            # ax.set_ylabel(f'Dim {i + 1}')
            ax.legend()
            ax.grid(True)
        idy += 1
    axes[-1].set_xlabel('alpha')
    return fig, axes


def plot_latent_reps(all_latents, model_dir, i_train):
    # Transpose the data to separate each joint
    all_latent_transp = []
    for latent_reps in all_latents:
        lat_rep_transp = list(zip(*latent_reps))
        all_latent_transp.append(lat_rep_transp)
    
    colors = [
        'blue', 'green', 'red', 'cyan', 'magenta', #'yellow', 
        'black', 'orange', 'purple', 'brown', 'pink', 'lime', 
        'teal', 'gold'
    ]
    fig_1, axes_1 = form_plot(all_latent_transp, 0, 7, colors)
    save_plot(model_dir, i_train, 'jnt')
    fig_2, axes_2 = form_plot(all_latent_transp, 7, 14, colors)
    save_plot(model_dir, i_train, 'vel')
    

def save_plot(model_dir, i_train, plt_name):
    plot_dir = os.path.join(model_dir, f'latent_reps/alpha_0')
    # print("plot_dir", plot_dir)
    if os.path.exists(plot_dir):
        pass
        # print("Latent dir. already exists. Are you trying to generate it again?")
        # sys.exit(1)
    else:
        os.makedirs(plot_dir)
    plot_path = os.path.join(plot_dir, 
                             f"train:{i_train}_{plt_name}.png")
    plt.savefig(plot_path)
    plt.close()

In [ ]:
epoch_no = 4000

for model_complexity in ['low', 'medium', 'high']:
    tester.load_model(0, 4000, 
                      tester.config.use_custom_loss, model_complexity)
    num_params = tester.config.get_params_num(tester.model)
    model_dir = os.path.join(root_dir, f"{ds_name}_{num_params}K_{ds_type}/ep:{episode_no}/")
    for i_train in range(10):
        tester.load_model(i_train, epoch_no, 
                          tester.config.use_custom_loss, model_complexity)

        all_latents = []
        last_ts_oneh = None
        for timestep_no in [0, 70, 140, 210, 290]:
            latent_reps = []
            for i_eps in range(tester.dataset.shape[0]):
                # timestep_no = 290
                elem = tester.dataset[i_eps]
                state = torch.tensor(elem[timestep_no][step_size : 
                                                step_size + state_size].tolist()
                                                ).to(device)

                # milestones = tester.get_changes_indexes(eps_num)
                one_hot = elem[timestep_no][step_size+state_size+target_size : 
                                    step_size+state_size+target_size+onehot_size].tolist()
                
                if last_ts_oneh == one_hot:
                    raise Exception("Timestep onehot match!")
                if i_eps == 0:
                    last_eps_oneh = one_hot
                else:
                    if one_hot != last_eps_oneh:
                        raise Exception("Episode one hot don't match.")
                    
                goal = elem[timestep_no][step_size+state_size : 
                                step_size+state_size+target_size].tolist()
                goal_tensor = torch.tensor(goal+one_hot).to(device)
                goal_nn = torch.unsqueeze(goal_tensor, 0)
                state_nn = torch.unsqueeze(state, 0)

                tester.model.eval()
                velocities_tensor, x_des, _ = tester.model(goal_nn, state_nn)
                x_des = torch.squeeze(x_des, 0)

                # print(goal)
                # print(one_hot)
                # print(x_des.tolist())
                # print("===")
                latent_reps.append(x_des.tolist())

            all_latents.append(latent_reps)
            last_ts_oneh = one_hot
            print(one_hot)
            print("-"*3)
        
        print("model_dir", model_dir)
        print("="*10)
        plot_latent_reps(all_latents, model_dir, i_train)